# Вариант 3: Классификация новостных заголовков
# 1. Описание данных

В этом разделе вам необходимо подробно описать предоставленный набор данных. Описание должно включать:

*   **Источник данных**: Откуда были получены данные?
*   **Размерность данных**: Сколько объектов (строк) и признаков (столбцов) содержится в наборе данных?
*   **Типы признаков**: Перечислите все признаки и укажите их тип (например, текстовые данные, категориальные, числовые).
*   **Распределение целевой переменной**: Если применимо, опишите распределение классов целевой переменной (например, сбалансировано ли оно, есть ли явный перекос).
*   **Примеры данных**: Приведите несколько примеров записей из набора данных, чтобы наглядно продемонстрировать их структуру.
*   **Общие наблюдения**: Любые другие важные характеристики или предварительные выводы о данных, которые могут повлиять на последующие этапы анализа и моделирования.

Цель этого раздела — дать полное понимание структуры и содержания данных, с которыми предстоит работать.

In [ ]:
#Загрузка SMS Spam Collection Dataset
import numpy as np
import pandas as pd
import requests
import zipfile
import io

# URL официального датасета от UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"

# Скачиваем zip-файл
print("Скачивание датасета...")
response = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(response.content))
z.extractall("sms_spam")

print("Файлы распакованы.")

# Загружаем основной файл
data_path = "sms_spam/SMSSpamCollection"

df = pd.read_csv(
    data_path,
    sep="\t",
    names=["label", "text"],
    encoding='utf-8'
)

# Преобразуем метки: ham=0, spam=1
df["label_num"] = df["label"].map({"ham": 0, "spam": 1})

print("Датасет успешно загружен!")

df.head()


Скачивание датасета...
Файлы распакованы.
Датасет успешно загружен!


,label,text,label_num
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   label         5572 non-null   object
 1   text          5572 non-null   object
 2   label_num     5572 non-null   int64 
 3   cleaned       5572 non-null   object
 4   no_stopwords  5572 non-null   object
 5   final_text    5572 non-null   object
dtypes: int64(1), object(5)
memory usage: 261.3+ KB


# 2. Предобработка текста

### Задание:
В этом разделе мы выполним предобработку текстовых данных, чтобы подготовить их для дальнейшего анализа или моделирования. Это важный этап, который помогает улучшить качество данных и производительность алгоритмов.

Следующие шаги необходимо выполнить в рамках предобработки текста:

1.  **Лемматизация/Стемминг**: Примените один из этих методов для приведения слов к их нормальной форме (например, "бегу", "бежал", "бежит" -> "бежать"). Объясните свой выбор между лемматизацией и стеммингом.
2.  **Удаление стоп-слов**: Удалите общие, неинформативные слова (стоп-слова, такие как "и", "в", "на" и т.д.), которые не несут смысловой нагрузки для анализа. Укажите, какой список стоп-слов вы используете.
3.  **Приведение к нижнему регистру**: Преобразуйте весь текст в нижний регистр для унификации и предотвращения обработки одного и того же слова в разных регистрах как разных слов.
4.  **Удаление пунктуации и специальных символов**: Удалите знаки препинания, числа, специальные символы и любые другие небуквенные символы, которые могут мешать анализу текста.
5.  **Обработка пропущенных значений**: Если в текстовых данных присутствуют пропущенные значения, опишите и реализуйте стратегию их обработки (например, удаление строк, заполнение заглушкой).
6.  **Визуализация до и после**: Для наглядности покажите примеры текста до и после применения каждого этапа предобработки. Это поможет убедиться в корректности выполненных шагов.
7.  **Причины каждого шага**: Объясните, почему каждый из вышеперечисленных шагов предобработки важен для данной задачи и как он способствует улучшению качества данных или результатов анализа.

##Подготовка инструментов

In [ ]:
!pip install nltk spacy --quiet
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 78.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
import re
import pandas as pd
import spacy
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize


In [ ]:
nlp = spacy.load("en_core_web_sm")          # Лемматизатор spaCy
stop_words = set(stopwords.words("english"))  # Стоп-слова NLTK
stemmer = PorterStemmer()                     # Стеммер Porter (альтернатива)


##Очистка текста

In [ ]:
def clean_text(text):
    text = text.lower()                         # приведение к нижнему регистру
    text = re.sub(r"http\S+", "", text)         # удаление ссылок
    text = re.sub(r"\d+", "", text)             # удаление чисел
    text = re.sub(r"[^\w\s]", "", text)         # удаление пунктуации
    text = re.sub(r"\s+", " ", text).strip()    # удаление лишних пробелов
    return text


In [ ]:
def lemmatize_text(text): #лемматизация
    doc = nlp(text)
    return " ".join([token.lemma_ for token in doc])


In [ ]:
def stem_text(text):  #стемминнг
    tokens = word_tokenize(text)
    return " ".join([stemmer.stem(w) for w in tokens])


In [ ]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    filtered = [word for word in tokens if word not in stop_words]
    return " ".join(filtered)


In [ ]:
# Сколько пропущенных значений?
df.isnull().sum()


,0
label,0
text,0
label_num,0


In [ ]:
print("Пример исходного текста:")
print(df['text'].iloc[10])


Пример исходного текста:
I'm gonna be home soon and i don't want to talk about this stuff anymore tonight, k? I've cried enough today.


In [ ]:
df["cleaned"] = df["text"].apply(clean_text)
df["no_stopwords"] = df["cleaned"].apply(remove_stopwords)

# Выбор между ЛЕММАТИЗАЦИЕЙ и СТЕММИНГОМ
USE_LEMMA = True   # Если False → используем стемминг

if USE_LEMMA:
    df["final_text"] = df["no_stopwords"].apply(lemmatize_text)
else:
    df["final_text"] = df["no_stopwords"].apply(stem_text)


In [ ]:
print(df['cleaned'].iloc[10])

im gonna be home soon and i dont want to talk about this stuff anymore tonight k ive cried enough today


# 3. Представление текста

В этом разделе необходимо преобразовать предобработанный текст в числовое представление, которое может быть использовано моделями машинного обучения. Выберите один или несколько методов векторизации текста и обоснуйте свой выбор.

Возможные методы:

1.  **TF-IDF (Term Frequency-Inverse Document Frequency)**:
    *   Объясните принцип работы TF-IDF.
    *   Примените TF-IDF для векторизации вашего корпуса текстов.
    *   Покажите примеры векторов TF-IDF для нескольких документов.
    *   Объясните, как параметры TF-IDF (например, `ngram_range`, `max_features`) влияют на результат.

2.  **Word Embeddings (например, Word2Vec, GloVe, FastText)**:
    *   Объясните концепцию Word Embeddings (векторных представлений слов).
    *   Загрузите предобученные эмбеддинги или обучите свои (если объем данных позволяет).
    *   Примените эмбеддинги для получения векторных представлений для каждого слова в ваших текстах. Возможно, вам потребуется стратегия для агрегации эмбеддингов слов в эмбеддинг предложения/документа (например, усреднение).
    *   Покажите примеры векторных представлений для нескольких слов и/или документов.
    *   Объясните преимущества и недостатки использования Word Embeddings по сравнению с TF-IDF.

3.  **Контекстуализированные Embeddings (например, BERT, RoBERTa, GPT)**:
    *   Объясните, что такое контекстуализированные эмбеддинги и в чем их отличие от обычных Word Embeddings.
    *   Используйте предобученную модель (например, из библиотеки `transformers`) для получения эмбеддингов предложений/документов.
    *   Покажите примеры векторных представлений.
    *   Объясните, когда имеет смысл использовать такие сложные модели и какие у них ограничения.

**Важно:**
*   Обоснуйте ваш выбор метода (или методов) векторизации, исходя из характеристик данных и целей задачи.
*   Убедитесь, что полученные векторные представления готовы для подачи в модель машинного обучения.
*   Опишите размерность полученных векторов и общую стратегию представления текстов.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf = TfidfVectorizer(
    max_features=3000,       # менять: 500, 2000, 10000
    ngram_range=(1, 2),      # униграммы + биграммы
    min_df=2                 # игнорируем редкие слова
)

X_tfidf = tfidf.fit_transform(df["final_text"])


In [ ]:
print("Размерность TF-IDF матрицы:", X_tfidf.shape)


Размерность TF-IDF матрицы: (5572, 3000)


In [ ]:
example_idx = 5
vector = X_tfidf[example_idx].toarray()[0]

print("Исходный текст:")
print(df["final_text"].iloc[example_idx])
print("\nTF-IDF вектор (первые 20 значений):")
print(vector[:20])


Исходный текст:
freemsg hey darling week word back i d like fun still tb ok xxx std chgs send rcv

TF-IDF вектор (первые 20 значений):
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


подумайте почему здесь нули? Нужно ли все переделывать?


In [ ]:
# Создаем векторизатор
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 4),
    stop_words=None
)

# Обучаем TF-IDF на вашем предобработанном тексте
X_tfidf = tfidf.fit_transform(df['final_text'])

# Сохраняем имена признаков (слов)
feature_names = tfidf.get_feature_names_out()

# Берём TF-IDF вектор первого документа
vec = X_tfidf[0].toarray()[0]

# Индексы ненулевых значений
nonzero_indices = np.where(vec != 0)[0]

print("Количество ненулевых признаков:", len(nonzero_indices))

# Покажем первые 20 ненулевых признаков
for idx in nonzero_indices[:20]:
    print(f"{feature_names[idx]} : {vec[idx]}")


Количество ненулевых признаков: 11
available : 0.3375921543171166
bugis : 0.37505529257119
cine : 0.37505529257119
crazy : 0.34060525268660957
get : 0.15664720710955704
go : 0.16534195489421424
great : 0.2461616056628294
la : 0.36396485686235336
point : 0.3031421144325362
wat : 0.24662828422494443
world : 0.3017014077118761


## 4. Модель (классический ML)
### Задание:
Создать текстовую ячейку с подробным описанием задания для Модели №1 (классический ML).

1.  **Выбор модели**: Выберите и обоснуйте выбор одной или нескольких классических моделей машинного обучения для задачи классификации текста (например, Logistic Regression, SVM, Naive Bayes, Random Forest, Gradient Boosting). Объясните, почему выбранная модель подходит для данной задачи.
2.  **Обучение модели**: Обучите выбранную модель на подготовленных данных (векторных представлениях текста и целевой переменной).
3.  **Параметры модели**: Опишите ключевые параметры выбранной модели и объясните, как они были настроены (например, использование GridSearchCV/RandomizedSearchCV для подбора гиперпараметров).
4.  **Код и комментарии**: Предоставьте чистый и хорошо прокомментированный код для обучения и оценки модели.
5.  **Первичная оценка**: Проведите первичную оценку производительности модели на тестовом наборе данных с использованием базовых метрик (например, accuracy).
6.  **Выводы**: Сделайте краткие выводы о работе первой модели, ее сильных и слабых сторонах.

In [ ]:
# Модель №1: Multinomial Naive Bayes
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV

# Убедимся, что у нас есть X_tfidf, y, X_train, X_test, y_train, y_test
print("Форма TF-IDF:", X_tfidf.shape)

Форма TF-IDF: (5572, 5000)


In [ ]:
from sklearn.model_selection import train_test_split

# -----------------------------------------
# X — это предобработанный текст
# y — метка класса ("spam" / "ham")
# -----------------------------------------
X = df['final_text']
y = df["label"]

# -----------------------------------------
# Разбиение на train и test
# stratify=y — гарантирует одинаковое распределение классов
# random_state — для воспроизводимости
# -----------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Размер обучающей выборки:", len(X_train))
print("Размер тестовой выборки:", len(X_test))

# -----------------------------------------
# После разбиения — сразу векторизуем (если используем TF-IDF)
# -----------------------------------------
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Форма X_train_tfidf:", X_train_tfidf.shape)
print("Форма X_test_tfidf:", X_test_tfidf.shape)

Размер обучающей выборки: 4457
Размер тестовой выборки: 1115
Форма X_train_tfidf: (4457, 5000)
Форма X_test_tfidf: (1115, 5000)


In [ ]:
# 1. Определение модели
nb = MultinomialNB()

# 2. Подбор гиперпараметров
param_grid = {
    'alpha': [0.1, 0.3, 0.5, 1.0]
}

grid = GridSearchCV(
    nb,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train_tfidf, y_train)

print("Лучший параметр alpha:", grid.best_params_)
print("Лучшая точность на CV:", grid.best_score_)

Лучший параметр alpha: {'alpha': 0.1}
Лучшая точность на CV: 0.979134141615457


In [ ]:
# 3. Обучение финальной модели

best_nb = grid.best_estimator_
best_nb.fit(X_train_tfidf, y_train)

# 4. Предсказания на тесте
y_pred = best_nb.predict(X_test_tfidf)



In [ ]:
# 5. Оценка качества
accuracy = accuracy_score(y_test, y_pred)
print("\nТочность на тесте:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Точность на тесте: 0.9775784753363229

Classification Report:
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       966
        spam       0.98      0.85      0.91       149

    accuracy                           0.98      1115
   macro avg       0.98      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115


Confusion Matrix:
[[964   2]
 [ 23 126]]


# Данная часть выполнена с использованием ИИ, в виду того, что у меня не было под руками готовой модели с полным и корректным пайпланом для дальнейшего использования в курсе
Поэтому далее я переделываю весь ноутбук в корректном формате, для последующего сохранения модели и использования ее далее в рамках курса.

In [26]:
import urllib.request
import zipfile
import pandas as pd

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"

urllib.request.urlretrieve(url, "smsspamcollection.zip")

with zipfile.ZipFile("smsspamcollection.zip", "r") as z:
    z.extractall("sms_data")

df = pd.read_csv("sms_data/SMSSpamCollection", sep="\t", header=None, names=["label", "text"])

# Преобразуем метки в 0/1 (0 = ham, 1 = spam)
df["target"] = df["label"].map({"ham": 0, "spam": 1})

print(df.shape)
df.head()

(5572, 3)


,label,text,target
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["target"],
    test_size=0.2,
    random_state=42,
    stratify=df["target"]
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")

Train: 4457, Test: 1115


In [28]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        max_features=5000
    )),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, ngram_range=(1, 2),
                                 stop_words='english')),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [29]:
from sklearn.metrics import (
    classification_report, precision_recall_curve,
    f1_score, roc_auc_score
)
import numpy as np

# Вероятности класса "spam" на тесте
y_proba = pipeline.predict_proba(X_test)[:, 1]

# Стандартная оценка при пороге 0.5
y_pred_default = (y_proba >= 0.5).astype(int)
print("=== Порог 0.5 ===")
print(classification_report(y_test, y_pred_default))

# Подбор оптимального порога по F1
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)

best_idx = np.argmax(f1_scores[:-1])  # последний элемент precision/recall без соответствующего threshold
best_threshold = thresholds[best_idx]

print(f"\nЛучший порог по F1: {best_threshold:.4f}")

y_pred_best = (y_proba >= best_threshold).astype(int)
print("=== Оптимальный порог ===")
print(classification_report(y_test, y_pred_best))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

=== Порог 0.5 ===
              precision    recall  f1-score   support

           0       0.99      0.98      0.99       966
           1       0.90      0.92      0.91       149

    accuracy                           0.98      1115
   macro avg       0.94      0.95      0.95      1115
weighted avg       0.98      0.98      0.98      1115


Лучший порог по F1: 0.6394
=== Оптимальный порог ===
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       966
           1       0.98      0.88      0.93       149

    accuracy                           0.98      1115
   macro avg       0.98      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115

ROC-AUC: 0.9872024677977407


In [30]:
import sklearn

metadata = {
    "version": "1.0.0",
    "features": ["text"],          # порядок входных признаков, ожидаемых pipeline
    "threshold": round(float(best_threshold), 4),
    "target_names": {0: "ham", 1: "spam"},
    "sklearn_version": sklearn.__version__,
    "model_type": "LogisticRegression",
    "vectorizer": "TfidfVectorizer",
    "train_size": len(X_train),
    "test_size": len(X_test),
    "metrics": {
        "roc_auc": round(float(roc_auc_score(y_test, y_proba)), 4),
        "f1_score": round(float(f1_score(y_test, y_pred_best)), 4)
    }
}

metadata

{'version': '1.0.0',
 'features': ['text'],
 'threshold': 0.6394,
 'target_names': {0: 'ham', 1: 'spam'},
 'sklearn_version': '1.6.1',
 'model_type': 'LogisticRegression',
 'vectorizer': 'TfidfVectorizer',
 'train_size': 4457,
 'test_size': 1115,
 'metrics': {'roc_auc': 0.9872, 'f1_score': 0.9291}}

In [31]:
import joblib

bundle = {
    "pipeline": pipeline,
    "metadata": metadata
}

joblib.dump(bundle, "model_bundle.joblib")
print("Сохранено: model_bundle.joblib")

Сохранено: model_bundle.joblib


In [32]:
loaded_bundle = joblib.load("model_bundle.joblib")

loaded_pipeline = loaded_bundle["pipeline"]
loaded_metadata = loaded_bundle["metadata"]

print("Metadata:", loaded_metadata)

# Пример предсказания на новых данных
sample_texts = [
    "Congratulations! You won a free prize, call now!",
    "Hey, are we still meeting for lunch tomorrow?"
]

probas = loaded_pipeline.predict_proba(sample_texts)[:, 1]
predictions = (probas >= loaded_metadata["threshold"]).astype(int)

for text, proba, pred in zip(sample_texts, probas, predictions):
    label = loaded_metadata["target_names"][pred]
    print(f"[{label}] (proba={proba:.3f}) — {text}")

Metadata: {'version': '1.0.0', 'features': ['text'], 'threshold': 0.6394, 'target_names': {0: 'ham', 1: 'spam'}, 'sklearn_version': '1.6.1', 'model_type': 'LogisticRegression', 'vectorizer': 'TfidfVectorizer', 'train_size': 4457, 'test_size': 1115, 'metrics': {'roc_auc': 0.9872, 'f1_score': 0.9291}}
[spam] (proba=0.954) — Congratulations! You won a free prize, call now!
[ham] (proba=0.048) — Hey, are we still meeting for lunch tomorrow?
